In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

In [ ]:
sys.path.append('/home/evolvotron/code/pomap')

In [ ]:
import polars as pl
from core.interpreter import _fit, _collect_labels, _collect_leaves, _predict, _get_train_df_for_label, _get_test_df_for_label
from core.nodes import *

import datetime as dt

### Set up some test models

In [ ]:
from scipy import stats

In [ ]:
@dataclass
class TestModel:
    x_column: str
    value = None

    def fit(self, df: pl.DataFrame):
        self.value = df[self.x_column].mean()

    def predict(self, df: pl.DataFrame):
        return [self.value] * df.shape[0]


In [ ]:
model = Leaf(
    label='lr',
    factory = lambda: LinearRegression(x_column='x', y_column='y')
            )

### Test Data

In [ ]:
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1d',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

# Example 1: Rolling retrain

In [ ]:
model = Leaf(
    label='tester',
    factory = lambda: TestModel(x_column='x')
    )

In [ ]:
retrain_points = [
    dt.datetime(2025, 1, 3),
    dt.datetime(2025, 1, 5)
]

rolling_train = Lift(
    model,
    atomics = retrain_points,
    train_mask_for_label = lambda label: pl.col('ts') <= label,
    test_mask_for_label = lambda label: pl.col('ts') > label,
    name='RollingRetrain'
)

In [ ]:
fitted_models = _fit(rolling_train, df)
_predict(rolling_train, fitted_models, df)

### Example 2: Randomised Cross Validation

In [ ]:
# For this one we need a bigger test dataframe
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1m',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

In [ ]:
def random_assignment_expr(column, num_categories, seed=0) -> pl.Expr:
    return pl.col(column).hash(seed=seed).mod(num_categories)

train_mask_for_label = lambda fold_number: random_assignment_expr('ts', 5) == fold_number

randomised_cv = Lift(
    model,
    atomics = range(5),
    train_mask_for_label=train_mask_for_label,
    test_mask_for_label = lambda fold_number: ~train_mask_for_label(fold_number),
    name='RandomisedAllocation'
)

In [ ]:
for fold in range(5):
    label = Label(RandomisedAllocation=fold, leaf='tester')
    train_set_size =  _get_train_df_for_label(randomised_cv,  df, label).shape[0]/df.shape[0]
    test_set_size = _get_test_df_for_label(randomised_cv,  df, label).shape[0]/df.shape[0]
    print(f'Train Set Size: {train_set_size:.2f}, Test set size {test_set_size:.2f}'
    )

### Exampel 4

### Example 3: Ensembles

### Example 3: Stateful Training